In [1]:
import gzip
import pickle

with gzip.open('../../dataset/train_set.pkl', 'rb') as file:
    train_set = pickle.load(file)
with gzip.open('dataset/test_set.pkl', 'rb') as file:
    test_set = pickle.load(file)
    
# Calculate the weight for each class in the training set
# Count the number of samples in each class in the label top20
class_counts = train_set['top20'].value_counts()
class_weights = {0: class_counts[1] / class_counts[0], 1: class_counts[0] / class_counts[1]}
print("Class weights: ", class_weights)


Class weights:  {0: 0.16575509770603228, 1: 6.032996956591382}


In [2]:
import os
import time
import gzip
import pickle
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib
matplotlib.use('Agg')  # Disable interactive mode
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
from sklearn.metrics import confusion_matrix, classification_report

import Classifier

BATCH_SIZE = 15000
NUM_EPOCHS = 40

def main():
    print("GRID SEARCH")
    print("Loading data...")

    # Loading dataset
    with gzip.open('../../dataset/train_set.pkl', 'rb') as file:
        train_set = pickle.load(file)
    with gzip.open('../../dataset/test_set.pkl', 'rb') as file:
        test_set = pickle.load(file)

    print("Data loaded!")

    #seed = int(time.time() % 150)
    seed = 23 # 23 Best - Mid 76
    print("Seed: ", seed)
    torch.manual_seed(seed)

    # Preprocessing base
    print("Dataset processing...")
    train_set = train_set.dropna()
    test_set = test_set.dropna()

    # Features
    features = ['performance_wma', 'top20_consistency', 'races_dby_consistency']
    print("Features: ", features)

    # Creaing X, y
    x_train = train_set[features]
    y_train = train_set['top20']
    x_test = test_set[features]
    y_test = test_set['top20']
    
    # Take wights for each class
    num_positives = y_train.sum()
    num_negatives = len(y_train) - num_positives
    class_weights = torch.tensor([num_negatives / num_positives], dtype=torch.float32)
    print("Class weights: ", class_weights)

    print("Dataset processed!")
    print("Dataset shape: ", x_train.shape, y_train.shape, x_test.shape, y_test.shape)

    input("Press Enter to continue...")  # Optional

    # Select device
    device = "cuda:0" if torch.cuda.is_available() else "cpu"
    print("Device: ", device)

    # Convert to tensor
    print("Converting to tensor...")
    x_train_t = torch.tensor(x_train.values, dtype=torch.float32).to(device)
    y_train_t = torch.tensor(y_train.values, dtype=torch.float32).to(device)
    x_test_t = torch.tensor(x_test.values, dtype=torch.float32).to(device)
    y_test_t = torch.tensor(y_test.values, dtype=torch.float32).to(device)
    print("Converted to tensor!")

    # Nwetwork configurations
    network_configs = [
        # {
        #     "layers": [3, 8, 10, 15, 1],
        #     "activation": "tanh",
        #     "optimizer": "sgd",
        #     "penality": 0.0002,
        #     "momentum": 0.9,
        #     "learning_rate": 0.1
        # },
        # {
        #     "layers": [3, 8, 10, 15, 1],
        #     "activation": "relu",
        #     "optimizer": "sgd",
        #     "penality": 0.0002,
        #     "momentum": 0.9,
        #     "learning_rate": 0.1
        # },
        {
            "layers": [3, 55, 110, 1],
            "activation": "tanh",
            "optimizer": "sgd",
            "penality": 0.0002,
            "momentum": 0.8,
            "learning_rate": 0.008
        }
        # Add others configurations
    ]

    numberTest = len(network_configs)
    
    bestResults = []

    # Dataloader (shuffle=True only for training)
    print("Creating data loader...")
    data_loader_train = torch.utils.data.DataLoader(
        list(zip(x_train_t, y_train_t)),
        batch_size=BATCH_SIZE,
        shuffle=True
    )
    data_loader_test = torch.utils.data.DataLoader(
        list(zip(x_test_t, y_test_t)),
        batch_size=BATCH_SIZE,
        shuffle=False
    )
    print("Data loader created!")

    print("Start training...")

    for number, config in enumerate(network_configs):
        print(f"\nTest[{number+1}/{numberTest}] --> Start training...")

        layers = config['layers']
        activation = config['activation']
        optimizerName = config['optimizer']
        penality = config['penality']
        momentum = config['momentum']
        lr = config['learning_rate']

        print(f"Test[{number+1}/{numberTest}] --> Layers: {layers}, Activation: {activation}, "
              f"Optimizer: {optimizerName}, Penality: {penality}, Momentum: {momentum}, Learning-rate: {lr}")

        # Name of the folder
        testName = f"{layers}-{optimizerName}-{activation}-{penality}-{momentum}-{lr}"
        pathName = f'models/Cup-{testName}'
        bestResults.append(testName + "\n")

        # Create directory
        print(f"Test[{number+1}/{numberTest}] --> Creating directory: {pathName}...")
        os.makedirs(pathName, exist_ok=True)
        print(f"Test[{number+1}/{numberTest}] --> Directory created!")

        # Create net
        print(f"Test[{number+1}/{numberTest}] --> Creating net...")
        structureNet = []  # if you want to add dropout, add it here
        net = Classifier.Classifier(layers, structureNet, activation, seed)
        net = net.to(device)
        net = net.float()
        print(f"Test[{number+1}/{numberTest}] --> Net created!")
        #
        class_weights = class_weights.to(device)
        # OPTIMIZER
        criterion = nn.BCEWithLogitsLoss()
        #criterion = nn.BCEWithLogitsLoss(pos_weight=class_weights)
        if optimizerName == "adam":
            optimizer = optim.Adam(net.parameters(), lr=lr, weight_decay=penality)
        elif optimizerName == "sgd":
            optimizer = optim.SGD(net.parameters(), lr=lr, momentum=momentum, weight_decay=penality)
        else:
            print("OPTIMIZER NOT FOUND!")
            exit(1)

        net.train()
        loss_values_train = []

        # TRAIN LOOP
        for epoch in range(NUM_EPOCHS):
            total_loss = 0.0
            for batch_input, batch_output in data_loader_train:
                # Forward
                outputs = net(batch_input)
                loss = criterion(outputs, batch_output.view(-1, 1))
                total_loss += loss.item()

                # Backward
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

            avg_loss_train = total_loss / len(data_loader_train)
            loss_values_train.append(avg_loss_train)
            print(f"Test[{number+1}/{numberTest}] --> "
                  f"Epoch[{epoch+1}/{NUM_EPOCHS}] --> Loss: {avg_loss_train:.6f}")

        print(f"Test[{number+1}/{numberTest}] --> Training finished!")
        torch.save(net.state_dict(), f"{pathName}/model.pth")
        # ----------------------------------------------------
        #  INFERENCE (prediction) WITHOUT SHUFFLE
        # ----------------------------------------------------
        net.eval()
        with torch.no_grad():
            pred_probs_train = torch.sigmoid(net(x_train_t)).cpu().numpy().ravel()
            pred_probs_test  = torch.sigmoid(net(x_test_t)).cpu().numpy().ravel()

        # ----------------------------------------------------
        #  REBUILD DATAFRAME CON PREDICTED_TOP20
        #  AND CALCULATE CONFUSION MATRIX AND CLASSIFICATION REPORT
        # ----------------------------------------------------
        train_df = train_set.copy()  # DataFrame with same rows/order of x_train
        train_df['pred_prob'] = pred_probs_train

        test_df = test_set.copy()    # DataFrame with same rows/order of x_test
        test_df['pred_prob'] = pred_probs_test

        # Method to evaluate top 20
        def evaluate_top_20(df, race_id_col='race_id', prob_col='pred_prob'):
            """
            Return DataFrame with column 'predicted_top20' set to
            1 for the top 20 rows of each group, 0 otherwise.
            """
            grouped_dfs = []
            for race_id, group in df.groupby(race_id_col):
                group = group.sort_values(by=prob_col, ascending=False).copy()
                group['predicted_top20'] = 0
                top_n = min(20, len(group))
                group.iloc[:top_n, group.columns.get_loc('predicted_top20')] = 1
                grouped_dfs.append(group)
            return pd.concat(grouped_dfs)
        #
        train_df = evaluate_top_20(train_df)
        test_df  = evaluate_top_20(test_df)

        # ----------------------------------------------------
        #  CALCULATE CONFUSION MATRIX AND CLASSIFICATION REPORT
        # ----------------------------------------------------
        y_true_train = train_df['top20'].values
        y_pred_train = train_df['predicted_top20'].values

        y_true_test  = test_df['top20'].values
        y_pred_test  = test_df['predicted_top20'].values

        cm_train = confusion_matrix(y_true_train, y_pred_train)
        cr_train = classification_report(y_true_train, y_pred_train)
        cm_test = confusion_matrix(y_true_test, y_pred_test)
        cr_test = classification_report(y_true_test, y_pred_test)

        print("\nConfusion Matrix - Train\n", cm_train)
        print("Classification Report - Train\n", cr_train)

        print("Confusion Matrix - Test\n", cm_test)
        print("Classification Report - Test\n", cr_test)

        # Save confusion matrix and classification report
        plt.figure(figsize=(10, 7))
        plt.title('Confusion Matrix - Train')
        sns.heatmap(cm_train, annot=True, fmt='d', cmap='Blues')
        plt.xlabel('Predicted')
        plt.ylabel('Actual')
        plt.savefig(f'{pathName}/Confusion-Matrix-Train.png')
        plt.clf()

        plt.figure(figsize=(10, 7))
        plt.title('Confusion Matrix - Test')
        sns.heatmap(cm_test, annot=True, fmt='d', cmap='Blues')
        plt.xlabel('Predicted')
        plt.ylabel('Actual')
        plt.savefig(f'{pathName}/Confusion-Matrix-Test.png')
        plt.clf()

        # Save classification report to file
        with open(f"{pathName}/scores-train.txt", "w") as file:
            file.write(cr_train)

        with open(f"{pathName}/scores-test.txt", "w") as file:
            file.write(cr_test)
            
        # Print accuracy - Train
        print(f"Test[{number+1}/{numberTest}] --> Accuracy Train: {cm_train[0][0] + cm_train[1][1]} / {len(y_true_train)} = {(cm_train[0][0] + cm_train[1][1]) / len(y_true_train)}")
        # F1 Score - Train
        print(f"Test[{number+1}/{numberTest}] --> F1 Score Train: {2 * (cm_train[0][0] / (cm_train[0][0] + cm_train[0][1]) * cm_train[0][0] / (cm_train[0][0] + cm_train[1][0])) / (cm_train[0][0] / (cm_train[0][0] + cm_train[0][1]) + cm_train[0][0] / (cm_train[0][0] + cm_train[1][0]))}")
        # Print accuracy - Test
        print(f"Test[{number+1}/{numberTest}] --> Accuracy Test: {cm_test[0][0] + cm_test[1][1]} / {len(y_true_test)} = {(cm_test[0][0] + cm_test[1][1]) / len(y_true_test)}")
        # F1 Score - Test
        print(f"Test[{number+1}/{numberTest}] --> F1 Score Test: {2 * (cm_test[0][0] / (cm_test[0][0] + cm_test[0][1]) * cm_test[0][0] / (cm_test[0][0] + cm_test[1][0])) / (cm_test[0][0] / (cm_test[0][0] + cm_test[0][1]) + cm_test[0][0] / (cm_test[0][0] + cm_test[1][0]))}")
        # Save accuracy
        bestResults.append(f"Accuracy Train: {cm_train[0][0] + cm_train[1][1]} / {len(y_true_train)} = {(cm_train[0][0] + cm_train[1][1]) / len(y_true_train)}\n")
        bestResults.append(f"Accuracy Test: {cm_test[0][0] + cm_test[1][1]} / {len(y_true_test)} = {(cm_test[0][0] + cm_test[1][1]) / len(y_true_test)}\n")
        # Save F1 Score
        bestResults.append(f"F1 Score Train: {2 * (cm_train[0][0] / (cm_train[0][0] + cm_train[0][1]) * cm_train[0][0] / (cm_train[0][0] + cm_train[1][0])) / (cm_train[0][0] / (cm_train[0][0] + cm_train[0][1]) + cm_train[0][0] / (cm_train[0][0] + cm_train[1][0]))}\n")
        bestResults.append(f"F1 Score Test: {2 * (cm_test[0][0] / (cm_test[0][0] + cm_test[0][1]) * cm_test[0][0] / (cm_test[0][0] + cm_test[1][0])) / (cm_test[0][0] / (cm_test[0][0] + cm_test[0][1]) + cm_test[0][0] / (cm_test[0][0] + cm_test[1][0]))}\n")

        print(f"Test[{number+1}/{numberTest}] --> End training!\n")
        
    # Save best results
    with open("bestResults.txt", "w") as file:
        for result in bestResults:
            file.write(result)

if __name__ == "__main__":
    main()

GRID SEARCH
Loading data...
Data loaded!
Seed:  23
Dataset processing...
Features:  ['performance_wma', 'top20_consistency', 'races_dby_consistency']
Class weights:  tensor([5.9876])
Dataset processed!
Dataset shape:  (426364, 3) (426364,) (34509, 3) (34509,)
Device:  cuda:0
Converting to tensor...
Converted to tensor!
Creating data loader...
Data loader created!
Start training...

Test[1/1] --> Start training...
Test[1/1] --> Layers: [3, 55, 110, 1], Activation: tanh, Optimizer: sgd, Penality: 0.0002, Momentum: 0.8, Learning-rate: 0.008
Test[1/1] --> Creating directory: models/Cup-[3, 55, 110, 1]-sgd-tanh-0.0002-0.8-0.008...
Test[1/1] --> Directory created!
Test[1/1] --> Creating net...
Test[1/1] --> Net created!
Test[1/1] --> Epoch[1/40] --> Loss: 0.399751
Test[1/1] --> Epoch[2/40] --> Loss: 0.377291
Test[1/1] --> Epoch[3/40] --> Loss: 0.374904
Test[1/1] --> Epoch[4/40] --> Loss: 0.373871
Test[1/1] --> Epoch[5/40] --> Loss: 0.372878
Test[1/1] --> Epoch[6/40] --> Loss: 0.372252
Test[1

In [4]:
# Model analysis
model = Classifier.Classifier([3, 55, 110, 1], [], 'tanh', 23)
model.load_state_dict(torch.load('model.pth'))
model.eval()

Classifier(
  (layers): ModuleList(
    (0): Linear(in_features=3, out_features=55, bias=True)
    (1): Linear(in_features=55, out_features=110, bias=True)
    (2): Linear(in_features=110, out_features=1, bias=True)
  )
)

In [10]:
# Test the model
# Features = 'performance_wma', 'top20_consistency', 'races_dby_consistency'

#walter-planckaert 9.333333333333334 2.0 1.0 -> 1 # tour-de-france/1978/stage-1a
test_input = torch.tensor([[9.333333333333334, 2.0, 1.0]], dtype=torch.float32)

print ('walter-planckaert 9.333333333333334 2.0 1.0 -> 1 # tour-de-france/1978/stage-1a')
print ('Name: walter-planckaert')
print ('Performance_wma: 9.333333333333334')
print ('Top20 consistency: 2.0')
print ('race_dby_consistency: 1.0')
print ('Top20 true: 1')
print ('Top20 predicted probability: ')
print(torch.sigmoid(model(test_input)))


#bernard-thevenet 29.666666666666664 1.0 2.0 -> 0 # tour-de-france/1978/stage-1a
test_input = torch.tensor([[92.33333333333334, 0.0, 2.0]], dtype=torch.float32)

print('bernard-thevenet 29.666666666666664 1.0 2.0 -> 0 # tour-de-france/1978/stage-1a')
print ('Name: bernard-thevenet')
print ('Performance_wma: 29.666666666666664')
print ('Top20 consistency: 1.0')
print ('races_dby_consistency: 2.0')
print ('Top20 true: 0')
print ('Top20 predicted probability: ')
print(torch.sigmoid(model(test_input)))

walter-planckaert 9.333333333333334 2.0 1.0 -> 1 # tour-de-france/1978/stage-1a
Name: walter-planckaert
Performance_wma: 9.333333333333334
Top20 consistency: 2.0
race_dby_consistency: 1.0
Top20 true: 1
Top20 predicted probability: 
tensor([[0.5196]], grad_fn=<SigmoidBackward0>)
bernard-thevenet 29.666666666666664 1.0 2.0 -> 0 # tour-de-france/1978/stage-1a
Name: bernard-thevenet
Performance_wma: 29.666666666666664
Top20 consistency: 1.0
races_dby_consistency: 2.0
Top20 true: 0
Top20 predicted probability: 
tensor([[0.1177]], grad_fn=<SigmoidBackward0>)


In [8]:
# Test the model
# Features = 'performance_wma', 'top20_consistency', 'races_dby_consistency'

#bernard-hinault 16.0 1.0 2.0 -> 0 # tour-de-france/1978/stage-1a
test_input = torch.tensor([[16.0, 1.0, 2.0]], dtype=torch.float32)

print ('bernard-hinault 16.0 1.0 2.0 -> 0 # tour-de-france/1978/stage-1a')
print ('Name: bernard-hinault')
print ('Performance_wma: 16.0')
print ('Top20 consistency: 1.0')
print('race_dby_consistency: 2.0')
print ('Top20 true: 0')
print ('Top20 predicted probability: ')
print(torch.sigmoid(model(test_input)))


#ludo-loos 92.33333333333334 0.0 2.0 -> 1 # tour-de-france/1978/stage-1a
test_input = torch.tensor([[92.33333333333334, 0.0, 2.0]], dtype=torch.float32)

print('ludo-loos 92.33333333333334 0.0 2.0 -> 1 # tour-de-france/1978/stage-1a')
print ('Name: ludo-loos')
print ('Performance_wma: 92.33333333333334')
print ('Top20 consistency: 0.0')
print ('race_dby_consistency: 2.0')
print ('Top20 true: 1')
print ('Top20 predicted probability: ')
print(torch.sigmoid(model(test_input)))

bernard-hinault 16.0 1.0 2.0 -> 0 # tour-de-france/1978/stage-1a
tensor([[0.4003]], grad_fn=<SigmoidBackward0>)
ludo-loos 92.33333333333334 0.0 2.0 -> 1 # tour-de-france/1978/stage-1a
tensor([[0.1177]], grad_fn=<SigmoidBackward0>)
